# Try loop-autopsy: every feature

A real Jupyter notebook running in your browser. Press **Shift + Enter** on each cell, or use **Run → Run All Cells**.

The first cell installs loop-autopsy 0.4.0 from PyPI (about 20 seconds the first time).

## 1. Install and turn it on

In [ ]:
%pip install -q loop-autopsy==0.4.0

# The browser cannot start threads, and loop-autopsy's watchdog runs in one.
# Every check that runs inside a loop round still works, so the watchdog is switched off here.
import threading
threading.Thread.start = lambda self: None

%load_ext loop_autopsy

## 2. A stuck loop
`count` never changes, so `count <= 5` is always True.

In [ ]:
count = 1
while count <= 5:
    print(count)

## 3. A loop that jumps over its stop value
`x` goes 0, 3, 6, 9, 12 and never equals 10.

In [ ]:
x = 0
while x != 10:
    x += 3

## 4. A list that keeps growing
The loop adds to the list it is reading, so it never reaches the end.

In [ ]:
nums = [1, 2]
for n in nums:
    nums.append(n)

## 5. Reading a report: a counter moving the wrong way
Every report has the same parts: what kind of bug and which line, why the loop can never end, the **replay table** of the last rounds, and a **hint** for the fix.

In [ ]:
i = 10
while i > 0:
    i += 1

## 6. The cell health bar
Time, memory and loops for a cell, coloured green, yellow or red. `auto` shows it only when a cell is slow or heavy; `on` shows it for every cell.

In [ ]:
%autopsy heatmap on
data = list(range(2_000_000))
total = 0
for v in data:
    total += v

In [ ]:
%autopsy heatmap auto

---
# More cases

## 7. Values going round in a cycle

In [ ]:
a = 0
while a < 5:
    a = (a + 1) % 3

## 8. Runaway recursion
Instead of Python's very long traceback: the first and last calls, and why the base case is never reached.

In [ ]:
def f(n):
    return f(n)

f(3)

## 9. An iterator that never runs out
Stopped before it even starts.

In [ ]:
import itertools
numbers = list(itertools.count())

## 10. A correct loop is left alone

In [ ]:
total = 0
for i in range(1_000_000):
    total += i
print("done", total)

---
# Hints read from your code

## 11. The update line comes after the loop

In [ ]:
count = 0
while count < 3:
    print(count)
count += 1

## 12. The update is inside an `if` that never runs

In [ ]:
i = 0
y = 2
while i < 5:
    if y > 3:
        i += 1

## 13. `while True:` with no way out
A note appears as soon as it starts.

In [ ]:
n = 0
while True:
    pass

## 14. A bare `except:` also catches the stop button

In [ ]:
total = 0
for k in range(3):
    try:
        total += k
    except:
        pass
print(total)

---
# Time

## 15. A slow loop gets an early estimate
For this demo the time limit is 3 seconds (the default is 60).

In [ ]:
%autopsy config time_limit=3 estimate_after=1

In [ ]:
import time
for row in range(1000):
    time.sleep(0.01)

## 16. The time limit stops anything else
loop-autopsy cannot prove this loop is endless (it calls `random`), so the time limit stops it.

In [ ]:
import random
x = 0.0
while x != -1:
    x = random.random()

In [ ]:
%autopsy reset

*A round stuck inside one long call (such as `time.sleep(600)` or a network request with no timeout) is stopped by the watchdog thread at the time limit. The browser cannot run that thread, so try it in a notebook on your own computer.*

---
# Memory and output

## 17. Memory in use and the biggest variables
In a notebook on your computer this also shows memory use against your limit; a browser cannot report it, so here it lists variable sizes.

In [ ]:
big = [0] * 1_000_000
%autopsy memory 3

## 18. The output guard
A cell is muted after 1,000 printed lines, so the browser does not freeze.

In [ ]:
for i in range(2000):
    print("line", i)

## 19. Session summary: the heaviest cells

In [ ]:
%autopsy summary

## 20. Show the last report again

In [ ]:
%autopsy last

---
# Settings and switches

## 21. Status and every setting

In [ ]:
%autopsy

## 22. Run one cell without watching

In [ ]:
%%autopsy off
q = 0
while q < 1000:
    q += 1
print(q)

## 23. Turn it off and on

In [ ]:
%autopsy off

In [ ]:
%autopsy on

---
# Optional extensions

## 24. `%%growthcheck`: preview a growing list without running the cell

In [ ]:
%load_ext loop_autopsy.growth_check

In [ ]:
%%growthcheck
items = [1, 2]
for item in items:
    items.append(item)

## 25. `%growthdiff`: which variables grew while a cell ran

In [ ]:
%load_ext loop_autopsy.growth_diff
samples = bytearray(1024)
%growthdiff mark

In [ ]:
samples.extend(bytes(4096))
%growthdiff show

## 26. `%memaliases` and `%memparts`: who holds a big object, and what is inside it

In [ ]:
%load_ext loop_autopsy.memory_inspect
batch = {"metadata": "train", "pixels": bytearray(8192)}
preview = batch
%memaliases batch

In [ ]:
%memparts batch

## 27. Extras: translated labels, a sparkline, export

In [ ]:
from loop_autopsy.extras.report_i18n import message, available_languages
print(available_languages())
for lang in available_languages():
    print(lang, "→", message("last_rounds", lang), "|", message("hint", lang))

In [ ]:
from loop_autopsy.extras.sparkline import sparkline_line
print(sparkline_line("x", [0, 3, 6, 9, 12, 15, 18], target=10))

In [ ]:
from loop_autopsy.extras.report_export import export_text
path = export_text("Loop autopsy — stuck loop (line 2)", "report.txt")
print("saved", path, "→", open(path).read())

## 28. All commands

In [ ]:
%autopsy help

---
On your own computer: `pip install loop-autopsy`, then `%load_ext loop_autopsy` in the first cell of any notebook.

Website: https://geoeoeo217.github.io/loop-autopsy-demo/